# Phase-Length / Break-Even Campaign
CPU-only, frozen 525-run V2 campaign. Accelerator must remain None.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, zipfile
root = Path('/kaggle/working/SIGMOD')
if root.exists(): shutil.rmtree(root)
bundles = list(Path('/kaggle/input').rglob('kaggle_upload_phase_length_v2.zip'))
if len(bundles) == 1:
    root.mkdir(parents=True)
    with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(root)
else:
    sources = [p.parent for p in Path('/kaggle/input').rglob('campaign_git_commit.txt') if (p.parent/'experiments/configs/phase_length_v2.json').is_file()]
    assert not bundles and len(sources) == 1, f'Expected one ZIP or extracted campaign dataset; ZIPs={len(bundles)}, sources={len(sources)}'
    shutil.copytree(sources[0], root)
required = [root/'experiments/configs/phase_length_v2_oracle_grid.json', root/'experiments/configs/phase_length_v2.json', root/'scripts/linux/run_phase_length_v2_oracle.sh', root/'scripts/linux/run_phase_length_v2.sh']
assert all(p.is_file() for p in required), [str(p) for p in required if not p.is_file()]
commit = (root/'campaign_git_commit.txt').read_text().strip()
assert len(commit) >= 7 and all(c in '0123456789abcdefABCDEF' for c in commit), commit
os.environ['CABTREE_GIT_COMMIT'] = commit
os.environ['CABTREE_CXXFLAGS'] = '-std=c++11 -O3 -march=native -DNDEBUG'
print('source commit:', commit)
print('root:', root)

In [ ]:
subprocess.run(['uname', '-a'], check=True)
subprocess.run(['lscpu'], check=True)
subprocess.run(['free', '-h'], check=True)
subprocess.run(['g++', '--version'], check=True)

In [ ]:
subprocess.run(['bash', str(root/'scripts/linux/build_release.sh'), '--execute'], cwd=root, check=True, env=os.environ.copy())
subprocess.run([str(root/'build/prepared/test_v2')], cwd=root, check=True, env=os.environ.copy())

In [ ]:
check = subprocess.run(['bash', str(root/'scripts/linux/run_phase_length_v2_oracle.sh'), 'dry-run', 'kaggle'], cwd=root, check=True, capture_output=True, text=True, env=os.environ.copy())
assert 'planned runs: 150' in check.stdout, check.stdout[-2000:]
print('Stage A frozen dry-run count: 150')

In [ ]:
# Stage A: six-fanout regional grid. Safe resume skips completed rows.
subprocess.run(['bash', str(root/'scripts/linux/run_phase_length_v2_oracle.sh'), '--execute', 'kaggle'], cwd=root, check=True, env=os.environ.copy())

In [ ]:
oracle_csv = root/'results/processed/phase_length_v2_oracle.csv'
subprocess.run(['python3', str(root/'analysis/phase_oracle.py'), '--raw', str(root/'results/raw/kaggle'), '--family', 'phase_length_v2_oracle_grid', '--output', str(oracle_csv)], cwd=root, check=True, env=os.environ.copy())
import csv
with oracle_csv.open(newline='') as stream: oracle_rows = list(csv.DictReader(stream))
assert len(oracle_rows) == 25, len(oracle_rows)
print('Measured oracle schedules:', len(oracle_rows))

In [ ]:
# Stage B dry-run resolves every Perfect-V2 schedule and must report 375.
check = subprocess.run(['bash', str(root/'scripts/linux/run_phase_length_v2.sh'), 'dry-run', 'kaggle'], cwd=root, check=True, capture_output=True, text=True, env=os.environ.copy())
assert 'planned runs: 375' in check.stdout, check.stdout[-2000:]
print('Stage B frozen dry-run count: 375')

In [ ]:
# Stage B: matched STATIC-REGIONAL, PERFECT-V2, and ADAPT-V2.
subprocess.run(['bash', str(root/'scripts/linux/run_phase_length_v2.sh'), '--execute', 'kaggle'], cwd=root, check=True, env=os.environ.copy())

In [ ]:
processed = root/'results/processed/phase_length_v2'
subprocess.run(['python3', str(root/'analysis/phase_length_break_even.py'), '--raw-root', str(root/'results/raw/kaggle'), '--oracle-csv', str(oracle_csv), '--out-dir', str(processed)], cwd=root, check=True, env=os.environ.copy())

In [ ]:
# Final cell: downloadable evidence and a resume bundle.
results_zip = shutil.make_archive('/kaggle/working/phase_length_v2_results', 'zip', root/'results')
resume_root = Path('/kaggle/working/phase_length_v2_resume_bundle')
if resume_root.exists(): shutil.rmtree(resume_root)
shutil.copytree(root/'results', resume_root/'results')
shutil.copy2(root/'experiments/configs/phase_length_v2_oracle_grid.json', resume_root/'phase_length_v2_oracle_grid.json')
shutil.copy2(root/'experiments/configs/phase_length_v2.json', resume_root/'phase_length_v2.json')
shutil.copy2(root/'campaign_git_commit.txt', resume_root/'campaign_git_commit.txt')
resume_zip = shutil.make_archive('/kaggle/working/phase_length_v2_resume', 'zip', resume_root)
print(results_zip)
print(resume_zip)